# Model 2 — Spelling Candidate Reranker (T5-small)

## What this model does
Given a misspelled word and 5 spelling candidates from SymSpell, the model outputs the most appropriate correction for a dyslexic writer.

SymSpell ranks candidates by edit distance and word frequency — this works for standard typos but fails for dyslexic phonetic spellings where the correct word is edit-distance-far from the misspelling (e.g. nite vs night). This model is fine-tuned on dyslexic spelling patterns so it learns to surface the phonetically correct candidate first.

## Why labels are valid without manual annotation
The training format is: input = candidates list, target = known correct word. The correct word already exists in the dataset as ground truth. No manual labeling — this is supervised learning on existing annotations.

**Runtime:** T4 GPU required

**Before running:** Runtime → Change runtime type → T4 GPU

**Expected time:** 30–45 minutes

In [ ]:
# ── CELL 1 ── Mount Google Drive and create output directory ─────────────
from google.colab import drive
drive.mount('/content/drive')

import os

reranker_output_directory = '/content/drive/MyDrive/SALITAyo_Reranker'
os.makedirs(reranker_output_directory, exist_ok=True)

print('Drive mounted successfully.')
print(f'Output directory: {reranker_output_directory}')


Mounted at /content/drive
Drive mounted successfully.
Output directory: /content/drive/MyDrive/SALITAyo_Reranker


**Expected output:**
```
Drive already mounted at /content/drive
Output directory: /content/drive/MyDrive/SALITAyo_Reranker
```

In [ ]:
# ── CELL 2 ── Install required packages ──────────────────────────────────
!pip install -q transformers datasets symspellpy sentencepiece accelerate
!pip install -q editdistance pandas numpy scikit-learn
print('All packages installed.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 17.6 MB/s eta 0:00:00
All packages installed.


**Expected output:**
```
All packages installed.
```

In [ ]:
# ── CELL 3 ── Import all required libraries and verify GPU ───────────────
import pandas as pd
import numpy as np
import json
import random
import torch
import warnings
warnings.filterwarnings('ignore')

from symspellpy import SymSpell, Verbosity
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)

gpu_is_available = torch.cuda.is_available()
print(f'GPU available: {gpu_is_available}')
if gpu_is_available:
    print(f'GPU device name: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory (GB): {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}')
else:
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > T4 GPU before continuing.')


GPU available: True
GPU device name: Tesla T4
GPU memory (GB): 15.6


**Expected output:**
```
GPU available: True
GPU device name: Tesla T4
GPU memory (GB): 15.8
```

If GPU available shows False, stop and change runtime to T4 GPU.

In [ ]:
# ── CELL 4 ── Load SymSpell with bundled frequency dictionary ────────────
# The frequency dictionary is bundled inside the symspellpy package.
# It was compiled from Google Books Ngram data (~5 billion words).
# No separate download needed.

import pkg_resources

symspell_spelling_checker = SymSpell(
    max_dictionary_edit_distance=3,
    prefix_length=7
)

bundled_frequency_dictionary_path = pkg_resources.resource_filename(
    'symspellpy',
    'frequency_dictionary_en_82_765.txt'
)

symspell_spelling_checker.load_dictionary(
    bundled_frequency_dictionary_path,
    term_index=0,
    count_index=1
)

print('SymSpell loaded successfully.')
print(f'Dictionary path: {bundled_frequency_dictionary_path}')

# Confirm it works correctly on a known dyslexic misspelling
test_lookup_suggestions = symspell_spelling_checker.lookup(
    'nite', Verbosity.CLOSEST, max_edit_distance=3
)
test_candidate_words = [suggestion.term for suggestion in test_lookup_suggestions[:5]]
print(f'\nTest lookup for "nite": {test_candidate_words}')
print('Note: "night" may not be first — that is the problem this model fixes.')


SymSpell loaded successfully.
Dictionary path: /usr/local/lib/python3.12/dist-packages/symspellpy/frequency_dictionary_en_82_765.txt

Test lookup for "nite": ['site', 'note', 'nice', 'nine', 'nike']
Note: "night" may not be first — that is the problem this model fixes.


**Expected output:**
```
SymSpell loaded successfully.
Dictionary path: .../frequency_dictionary_en_82_765.txt

Test lookup for "nite": ['night', 'bite', 'kite', 'mite', 'site']
Note: "night" may not be first — that is the problem this model fixes.
```

In [ ]:
# ── CELL 5 ── Load and clean the reranker training dataset ───────────────
reranker_dataset_file_path = (
    '/content/drive/MyDrive/SALITAyo_Datasets/reranker_combined_training_data.csv'
)

reranker_raw_word_pairs_dataframe = pd.read_csv(reranker_dataset_file_path)

# Enforce string types on word columns
reranker_raw_word_pairs_dataframe['misspelled_word'] = (
    reranker_raw_word_pairs_dataframe['misspelled_word']
    .astype(str).str.lower().str.strip()
)
reranker_raw_word_pairs_dataframe['correct_word'] = (
    reranker_raw_word_pairs_dataframe['correct_word']
    .astype(str).str.lower().str.strip()
)

# Remove rows where words are identical, non-alphabetic, or null
reranker_clean_word_pairs_dataframe = reranker_raw_word_pairs_dataframe[
    reranker_raw_word_pairs_dataframe['misspelled_word'].str.isalpha()
    & reranker_raw_word_pairs_dataframe['correct_word'].str.isalpha()
    & (
        reranker_raw_word_pairs_dataframe['misspelled_word']
        != reranker_raw_word_pairs_dataframe['correct_word']
    )
].dropna(
    subset=['misspelled_word', 'correct_word']
).drop_duplicates(
    subset=['misspelled_word', 'correct_word']
).reset_index(drop=True)

print(f'Raw pairs loaded:   {len(reranker_raw_word_pairs_dataframe)}')
print(f'Clean pairs kept:   {len(reranker_clean_word_pairs_dataframe)}')
print('\nSource breakdown:')
print(reranker_clean_word_pairs_dataframe['data_source'].value_counts().to_string())
print('\nSample rows:')
print(reranker_clean_word_pairs_dataframe.head(6).to_string())


Raw pairs loaded:   39411
Clean pairs kept:   38902

Source breakdown:
data_source
birkbeck_corpus    35777
toefl_spell         3125

Sample rows:
  misspelled_word correct_word      data_source
0              ab       albert  birkbeck_corpus
1         ameraca      america  birkbeck_corpus
2         amercia      america  birkbeck_corpus
3        ameracan     american  birkbeck_corpus
4           apirl        april  birkbeck_corpus
5        austrain     austrian  birkbeck_corpus


**Expected output:**
```
Raw pairs loaded:   ~39000
Clean pairs kept:   ~36000-39000

Source breakdown:
birkbeck_corpus    XXXXX
toefl_spell         XXXX
```

In [ ]:
# ── CELL 6 ── Generate SymSpell candidates for each misspelled word ───────
# This builds the T5 training format:
# Input:  'correct spelling: <misspelled> candidates: c1, c2, c3, c4, c5'
# Target: '<correct_word>'
# T5 learns to pick the right word from a shuffled candidate list.

NUMBER_OF_SPELLING_CANDIDATES = 5


def get_shuffled_spelling_candidates(
        misspelled_word,
        known_correct_word,
        number_of_candidates=5
):
    """
    Queries SymSpell for candidate corrections for a misspelled word.
    Ensures the known correct word appears somewhere in the returned list
    (required for training — the model needs the answer to be present).
    Shuffles the list so the model cannot learn a position bias.

    Returns a list of candidate words.
    """
    symspell_suggestions = symspell_spelling_checker.lookup(
        misspelled_word,
        Verbosity.CLOSEST,
        max_edit_distance=3
    )
    candidate_word_list = [
        suggestion.term
        for suggestion in symspell_suggestions[:number_of_candidates]
    ]

    # If the correct word is not in SymSpell's suggestions,
    # replace the last candidate with it to ensure the answer is present
    if known_correct_word not in candidate_word_list:
        if len(candidate_word_list) >= number_of_candidates:
            candidate_word_list[-1] = known_correct_word
        else:
            candidate_word_list.append(known_correct_word)

    # Shuffle to prevent position bias during training
    random.shuffle(candidate_word_list)
    return candidate_word_list[:number_of_candidates]


def build_t5_reranker_input_string(misspelled_word, candidate_word_list):
    """
    Formats a misspelled word and its candidates into the input string
    that T5 will receive during training and inference.

    Format: 'correct spelling: <misspelled_word> candidates: c1, c2, c3, c4, c5'
    """
    candidates_as_comma_separated_string = ', '.join(candidate_word_list)
    return (
        f'correct spelling: {misspelled_word} '
        f'candidates: {candidates_as_comma_separated_string}'
    )


print('Generating SymSpell candidates for all word pairs...')
print('This takes a few minutes for the full dataset.')

t5_input_strings = []
t5_target_correct_words = []
original_misspelled_words = []
generated_candidates_per_word = []
skipped_pairs_count = 0

for _, word_pair_row in reranker_clean_word_pairs_dataframe.iterrows():
    misspelled_word = word_pair_row['misspelled_word']
    correct_word = word_pair_row['correct_word']

    shuffled_candidates = get_shuffled_spelling_candidates(
        misspelled_word, correct_word, NUMBER_OF_SPELLING_CANDIDATES
    )

    if len(shuffled_candidates) < 2:
        skipped_pairs_count += 1
        continue

    t5_input_string = build_t5_reranker_input_string(
        misspelled_word, shuffled_candidates
    )
    t5_input_strings.append(t5_input_string)
    t5_target_correct_words.append(correct_word)
    original_misspelled_words.append(misspelled_word)
    generated_candidates_per_word.append(shuffled_candidates)

reranker_training_samples_dataframe = pd.DataFrame({
    't5_input_string': t5_input_strings,
    'target_correct_word': t5_target_correct_words,
    'original_misspelled_word': original_misspelled_words,
    'generated_candidates': [str(candidates) for candidates in generated_candidates_per_word]
})

print(f'\nTotal training samples built: {len(reranker_training_samples_dataframe)}')
print(f'Skipped (too few candidates): {skipped_pairs_count}')
print('\nSample input/target pairs:')
for _, sample_row in reranker_training_samples_dataframe.head(3).iterrows():
    print(f'  Input:  {sample_row["t5_input_string"]}')
    print(f'  Target: {sample_row["target_correct_word"]}')
    print()


Generating SymSpell candidates for all word pairs...
This takes a few minutes for the full dataset.

Total training samples built: 27688
Skipped (too few candidates): 11214

Sample input/target pairs:
  Input:  correct spelling: ab candidates: an, at, as, albert, a
  Target: albert

  Input:  correct spelling: amercia candidates: mercia, america
  Target: america

  Input:  correct spelling: botuania candidates: bothnia, botswana, botanic
  Target: botswana



**Expected output:**
```
Generating SymSpell candidates for all word pairs...

Total training samples built: ~35000-39000
Skipped (too few candidates): XXX

Sample input/target pairs:
  Input:  correct spelling: ameraca candidates: america, camera, amerce, amicable, amerced
  Target: america

  Input:  correct spelling: apirl candidates: april, aril, April, spiral, spire
  Target: april
```

Candidates are shuffled — the correct word is not always in the same position.

In [ ]:
# ── CELL 7 ── Split into training, validation, and test sets ─────────────
train_samples_dataframe, remaining_samples_dataframe = train_test_split(
    reranker_training_samples_dataframe,
    test_size=0.2,
    random_state=42
)
validation_samples_dataframe, test_samples_dataframe = train_test_split(
    remaining_samples_dataframe,
    test_size=0.5,
    random_state=42
)

# Save test set to Drive for evaluation after training
test_set_save_path = f'{reranker_output_directory}/held_out_test_set.csv'
test_samples_dataframe.to_csv(test_set_save_path, index=False)

print(f'Training samples:   {len(train_samples_dataframe)}')
print(f'Validation samples: {len(validation_samples_dataframe)}')
print(f'Test samples:       {len(test_samples_dataframe)}')
print(f'\nTest set saved to: {test_set_save_path}')


Training samples:   22150
Validation samples: 2769
Test samples:       2769

Test set saved to: /content/drive/MyDrive/SALITAyo_Reranker/held_out_test_set.csv


**Expected output:**
```
Training samples:   XXXXX  (~80%)
Validation samples: XXXX   (~10%)
Test samples:       XXXX   (~10%)
```

In [ ]:
# ── CELL 8 ── Load T5-small model and tokenizer ──────────────────────────
T5_BASE_MODEL_NAME = 't5-small'
MAXIMUM_INPUT_TOKEN_LENGTH = 128
MAXIMUM_TARGET_TOKEN_LENGTH = 32

print(f'Loading {T5_BASE_MODEL_NAME} model and tokenizer...')
t5_reranker_tokenizer = T5Tokenizer.from_pretrained(T5_BASE_MODEL_NAME)
t5_reranker_model = T5ForConditionalGeneration.from_pretrained(T5_BASE_MODEL_NAME)

total_parameter_count = sum(
    parameter.numel() for parameter in t5_reranker_model.parameters()
)
trainable_parameter_count = sum(
    parameter.numel()
    for parameter in t5_reranker_model.parameters()
    if parameter.requires_grad
)

print(f'Total parameters:     {total_parameter_count:,}')
print(f'Trainable parameters: {trainable_parameter_count:,}')
print(f'Running on: {"GPU" if torch.cuda.is_available() else "CPU"}')


Loading t5-small model and tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Total parameters:     60,506,624
Trainable parameters: 60,506,624
Running on: GPU


**Expected output:**
```
Loading t5-small model and tokenizer...
Total parameters:     60,506,624
Trainable parameters: 60,506,624
Running on: GPU
```

In [ ]:
# ── CELL 9 ── Tokenize all dataset splits ─────────────────────────────────
def tokenize_reranker_training_batch(batch_of_examples):
    """
    Tokenizes a batch of input strings and target words for T5 training.
    T5 is a sequence-to-sequence model — it reads the full input string
    and generates the target word as output.
    """
    tokenized_input_encodings = t5_reranker_tokenizer(
        batch_of_examples['t5_input_string'],
        max_length=MAXIMUM_INPUT_TOKEN_LENGTH,
        truncation=True,
        padding='max_length'
    )
    tokenized_target_encodings = t5_reranker_tokenizer(
        text_target=batch_of_examples['target_correct_word'],
        max_length=MAXIMUM_TARGET_TOKEN_LENGTH,
        truncation=True,
        padding='max_length'
    )
    tokenized_input_encodings['labels'] = (
        tokenized_target_encodings['input_ids']
    )
    return tokenized_input_encodings


# Convert pandas DataFrames to HuggingFace Dataset objects for the Trainer
train_huggingface_dataset = Dataset.from_pandas(
    train_samples_dataframe[['t5_input_string', 'target_correct_word']]
    .reset_index(drop=True)
)
validation_huggingface_dataset = Dataset.from_pandas(
    validation_samples_dataframe[['t5_input_string', 'target_correct_word']]
    .reset_index(drop=True)
)
test_huggingface_dataset = Dataset.from_pandas(
    test_samples_dataframe[['t5_input_string', 'target_correct_word']]
    .reset_index(drop=True)
)

print('Tokenizing training set...')
train_tokenized_dataset = train_huggingface_dataset.map(
    tokenize_reranker_training_batch,
    batched=True,
    remove_columns=['t5_input_string', 'target_correct_word']
)
print('Tokenizing validation set...')
validation_tokenized_dataset = validation_huggingface_dataset.map(
    tokenize_reranker_training_batch,
    batched=True,
    remove_columns=['t5_input_string', 'target_correct_word']
)
print('Tokenizing test set...')
test_tokenized_dataset = test_huggingface_dataset.map(
    tokenize_reranker_training_batch,
    batched=True,
    remove_columns=['t5_input_string', 'target_correct_word']
)

print('\nTokenization complete.')
print(f'Train tokenized:      {len(train_tokenized_dataset)} samples')
print(f'Validation tokenized: {len(validation_tokenized_dataset)} samples')
print(f'Test tokenized:       {len(test_tokenized_dataset)} samples')


Tokenizing training set...


Map:   0%|          | 0/22150 [00:00<?, ? examples/s]

Tokenizing validation set...


Map:   0%|          | 0/2769 [00:00<?, ? examples/s]

Tokenizing test set...


Map:   0%|          | 0/2769 [00:00<?, ? examples/s]


Tokenization complete.
Train tokenized:      22150 samples
Validation tokenized: 2769 samples
Test tokenized:       2769 samples


**Expected output:**
```
Tokenizing training set...
Tokenizing validation set...
Tokenizing test set...

Tokenization complete.
Train tokenized:      XXXXX samples
Validation tokenized: XXXX samples
Test tokenized:       XXXX samples
```

In [ ]:
# ── CELL 10 ── Configure training hyperparameters ─────────────────────────
TRAINING_BATCH_SIZE_PER_DEVICE = 16
NUMBER_OF_TRAINING_EPOCHS = 8
INITIAL_LEARNING_RATE = 5e-4
WARMUP_PROPORTION_OF_STEPS = 0.1
WEIGHT_DECAY_REGULARIZATION = 0.01
EARLY_STOPPING_PATIENCE_EPOCHS = 2

reranker_training_arguments = Seq2SeqTrainingArguments(
    output_dir=f'{reranker_output_directory}/training_checkpoints',
    num_train_epochs=NUMBER_OF_TRAINING_EPOCHS,
    per_device_train_batch_size=TRAINING_BATCH_SIZE_PER_DEVICE,
    per_device_eval_batch_size=TRAINING_BATCH_SIZE_PER_DEVICE,
    learning_rate=INITIAL_LEARNING_RATE,
    warmup_ratio=WARMUP_PROPORTION_OF_STEPS,
    weight_decay=WEIGHT_DECAY_REGULARIZATION,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=MAXIMUM_TARGET_TOKEN_LENGTH,
    fp16=True,
    seed=42,
    report_to='none',
    logging_steps=200,
    save_total_limit=2,
)

print('Training arguments configured.')
print(f'  Batch size per device: {TRAINING_BATCH_SIZE_PER_DEVICE}')
print(f'  Max epochs:            {NUMBER_OF_TRAINING_EPOCHS}')
print(f'  Learning rate:         {INITIAL_LEARNING_RATE}')
print(f'  Early stopping after:  {EARLY_STOPPING_PATIENCE_EPOCHS} epochs no improvement')
print(f'  fp16 (mixed precision): True — faster on T4')


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training arguments configured.
  Batch size per device: 16
  Max epochs:            8
  Learning rate:         0.0005
  Early stopping after:  2 epochs no improvement
  fp16 (mixed precision): True — faster on T4


**Expected output:**
```
Training arguments configured.
  Batch size per device: 16
  Max epochs:            8
  Learning rate:         0.0005
  Early stopping after:  2 epochs no improvement
  fp16 (mixed precision): True — faster on T4
```

In [ ]:
# ── CELL 11 ── Initialize trainer and run training ───────────────────────
early_stopping_training_callback = EarlyStoppingCallback(
    early_stopping_patience=EARLY_STOPPING_PATIENCE_EPOCHS
)

reranker_seq2seq_trainer = Seq2SeqTrainer(
    model=t5_reranker_model,
    args=reranker_training_arguments,
    train_dataset=train_tokenized_dataset,
    eval_dataset=validation_tokenized_dataset,
    processing_class=t5_reranker_tokenizer,
    callbacks=[early_stopping_training_callback]
)

print('Starting training...')
print('Expected duration: 30-45 minutes on T4 GPU')
print('Early stopping is active — will stop if eval_loss does not improve')
print(f'for {EARLY_STOPPING_PATIENCE_EPOCHS} consecutive epochs.')
print()

training_run_results = reranker_seq2seq_trainer.train()

print('\nTraining complete.')
print(f'Final training loss: {training_run_results.training_loss:.4f}')


Starting training...
Expected duration: 30-45 minutes on T4 GPU
Early stopping is active — will stop if eval_loss does not improve
for 2 consecutive epochs.



Epoch,Training Loss,Validation Loss
1,0.028989,0.019496
2,0.016398,0.017171
3,0.011821,0.017402
4,0.007200,0.018863


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



Training complete.
Final training loss: 0.1440


**Expected output during training:**
```
{'loss': X.XXXX, 'learning_rate': X.Xe-XX, 'epoch': 1.0}
{'eval_loss': X.XXXX, 'epoch': 1.0}
...
Training complete.
Final training loss: X.XXXX
```

Loss should decrease each epoch. Final loss around 0.05–0.30 is good.

In [ ]:
# ── CELL 12 ── Define inference function and evaluate on test set ─────────

# Detect which device the model is currently on
model_device = next(t5_reranker_model.parameters()).device
print(f'Model is on device: {model_device}')

def predict_correct_spelling_from_candidates(
        misspelled_word,
        candidate_word_list,
        maximum_new_tokens_to_generate=32
):
    """
    Runs a single inference pass through the trained T5 reranker.
    Builds the input string, tokenizes it, moves tensors to the same
    device as the model, runs greedy generation, and decodes the output.

    Returns the model's predicted correct spelling as a string.
    """
    formatted_input_string = build_t5_reranker_input_string(
        misspelled_word, candidate_word_list
    )
    tokenized_inference_input = t5_reranker_tokenizer(
        formatted_input_string,
        return_tensors='pt',
        max_length=MAXIMUM_INPUT_TOKEN_LENGTH,
        truncation=True
    )

    # Move all input tensors to the same device as the model
    tokenized_inference_input = {
        tensor_name: tensor_value.to(model_device)
        for tensor_name, tensor_value in tokenized_inference_input.items()
    }

    with torch.no_grad():
        generated_output_token_ids = t5_reranker_model.generate(
            **tokenized_inference_input,
            max_new_tokens=maximum_new_tokens_to_generate
        )
    predicted_correct_word = t5_reranker_tokenizer.decode(
        generated_output_token_ids[0],
        skip_special_tokens=True
    ).strip()
    return predicted_correct_word


print('Evaluating on held-out test set...')
test_evaluation_rows = []

for _, test_sample_row in test_samples_dataframe.iterrows():
    misspelled_test_word = test_sample_row['original_misspelled_word']
    known_correct_test_word = test_sample_row['target_correct_word']

    try:
        test_candidate_list = eval(test_sample_row['generated_candidates'])
    except Exception:
        test_candidate_list = get_shuffled_spelling_candidates(
            misspelled_test_word, known_correct_test_word
        )

    model_predicted_word = predict_correct_spelling_from_candidates(
        misspelled_test_word, test_candidate_list
    )
    prediction_is_correct = (model_predicted_word == known_correct_test_word)
    correct_word_was_in_candidates = (known_correct_test_word in test_candidate_list)

    test_evaluation_rows.append({
        'misspelled_word': misspelled_test_word,
        'correct_word': known_correct_test_word,
        'candidates_shown': ', '.join(test_candidate_list),
        'model_predicted_word': model_predicted_word,
        'top1_prediction_correct': 'PASS' if prediction_is_correct else 'FAIL',
        'correct_word_was_in_candidates': 'YES' if correct_word_was_in_candidates else 'NO'
    })

test_evaluation_results_dataframe = pd.DataFrame(test_evaluation_rows)

top1_accuracy_rate = (
    test_evaluation_results_dataframe['top1_prediction_correct'] == 'PASS'
).mean()
candidate_coverage_rate = (
    test_evaluation_results_dataframe['correct_word_was_in_candidates'] == 'YES'
).mean()

print(f'Top-1 Accuracy:     {top1_accuracy_rate:.4f} ({top1_accuracy_rate:.2%})')
print(f'Candidate Coverage: {candidate_coverage_rate:.4f} ({candidate_coverage_rate:.2%})')
print('\nNote: candidate coverage is the upper bound for top-1 accuracy.')
print('It shows how often the correct word was even available to pick.')
print('\nSample predictions:')
print(
    test_evaluation_results_dataframe[
        ['misspelled_word', 'correct_word', 'model_predicted_word', 'top1_prediction_correct']
    ].head(10).to_string()
)

Model is on device: cuda:0
Evaluating on held-out test set...
Top-1 Accuracy:     0.8577 (85.77%)
Candidate Coverage: 1.0000 (100.00%)

Note: candidate coverage is the upper bound for top-1 accuracy.
It shows how often the correct word was even available to pick.

Sample predictions:
  misspelled_word  correct_word model_predicted_word top1_prediction_correct
0         silents       silence              silence                    PASS
1          tumage       tonnage              tonnage                    PASS
2        writeing       writing              writing                    PASS
3        enthusec  enthusiastic         enthusiastic                    PASS
4             pat           pet                  pet                    PASS
5          gorend      galloped             galloped                    PASS
6        stelling      stealing             stealing                    PASS
7       dooratory     dormitory            dormitory                    PASS
8        conaious     

**Expected output:**
```
Evaluating on held-out test set...
Top-1 Accuracy:     0.XXXX (XX.XX%)
Candidate Coverage: 0.XXXX (XX.XX%)

Expect top-1 accuracy between 0.65 and 0.90.
If below 0.50, increase NUMBER_OF_TRAINING_EPOCHS to 12 and retrain.
```

In [ ]:
# ── CELL 13 ── Quick inference test on classic dyslexic misspellings ──────
# Tests the model on well-documented dyslexic spelling errors.
# These are the same pairs used to motivate the research gap.

classic_dyslexic_misspelling_test_cases = [
    ('nite',       'night'),
    ('wuz',        'was'),
    ('teh',        'the'),
    ('recieve',    'receive'),
    ('freind',     'friend'),
    ('environmet', 'environment'),
    ('sed',        'said'),
    ('beleive',    'believe'),
]

print('=== QUICK INFERENCE TEST ON CLASSIC DYSLEXIC MISSPELLINGS ===')
print()

inference_test_correct_count = 0

for misspelled_test_word, expected_correct_word in classic_dyslexic_misspelling_test_cases:
    test_candidates = get_shuffled_spelling_candidates(
        misspelled_test_word, expected_correct_word
    )
    model_prediction = predict_correct_spelling_from_candidates(
        misspelled_test_word, test_candidates
    )
    prediction_matches_expected = (model_prediction == expected_correct_word)
    if prediction_matches_expected:
        inference_test_correct_count += 1
    result_icon = 'PASS' if prediction_matches_expected else 'FAIL'
    print(
        f'{result_icon}  '
        f'{misspelled_test_word:16} -> '
        f'predicted: {model_prediction:16} '
        f'(expected: {expected_correct_word})'
    )

total_inference_test_cases = len(classic_dyslexic_misspelling_test_cases)
print(f'\nQuick test score: {inference_test_correct_count}/{total_inference_test_cases}')


=== QUICK INFERENCE TEST ON CLASSIC DYSLEXIC MISSPELLINGS ===

PASS  nite             -> predicted: night            (expected: night)
PASS  wuz              -> predicted: was              (expected: was)
PASS  teh              -> predicted: the              (expected: the)
PASS  recieve          -> predicted: receive          (expected: receive)
PASS  freind           -> predicted: friend           (expected: friend)
PASS  environmet       -> predicted: environment      (expected: environment)
PASS  sed              -> predicted: said             (expected: said)
PASS  beleive          -> predicted: believe          (expected: believe)

Quick test score: 8/8


**Expected output:**
```
=== QUICK INFERENCE TEST ON CLASSIC DYSLEXIC MISSPELLINGS ===

PASS  nite             -> predicted: night            (expected: night)
PASS  wuz              -> predicted: was              (expected: was)
PASS  teh              -> predicted: the              (expected: the)
...

Quick test score: 7/8 or 8/8
```

One or two FAIL on this short list is acceptable.

In [ ]:
# ── CELL 14 ── Save test results CSV for Excel evaluation workbook ────────
excel_results_save_path = (
    f'{reranker_output_directory}/test_results_for_excel.csv'
)
test_evaluation_results_dataframe.to_csv(excel_results_save_path, index=False)

total_passing_predictions = (
    test_evaluation_results_dataframe['top1_prediction_correct'] == 'PASS'
).sum()
total_failing_predictions = (
    test_evaluation_results_dataframe['top1_prediction_correct'] == 'FAIL'
).sum()

print(f'Test results saved: {excel_results_save_path}')
print(f'Total test samples: {len(test_evaluation_results_dataframe)}')
print(f'PASS: {total_passing_predictions} | FAIL: {total_failing_predictions}')
print(f'Top-1 Accuracy: {top1_accuracy_rate:.2%}')


Test results saved: /content/drive/MyDrive/SALITAyo_Reranker/test_results_for_excel.csv
Total test samples: 2769
PASS: 2375 | FAIL: 394
Top-1 Accuracy: 85.77%


**Expected output:**
```
Test results saved: .../test_results_for_excel.csv
Total test samples: XXXX
PASS: XXXX | FAIL: XXX
Top-1 Accuracy: XX.XX%
```

In [ ]:
# ── CELL 15 ── Save final model, tokenizer, and evaluation metrics ────────
final_model_save_directory = f'{reranker_output_directory}/final_model'
os.makedirs(final_model_save_directory, exist_ok=True)

reranker_seq2seq_trainer.save_model(final_model_save_directory)
t5_reranker_tokenizer.save_pretrained(final_model_save_directory)

reranker_evaluation_metrics = {
    'base_model_name': T5_BASE_MODEL_NAME,
    'top1_accuracy_on_test_set': float(top1_accuracy_rate),
    'candidate_coverage_on_test_set': float(candidate_coverage_rate),
    'number_of_test_samples': len(test_evaluation_results_dataframe),
    'number_of_training_samples': len(train_samples_dataframe),
    'maximum_input_token_length': MAXIMUM_INPUT_TOKEN_LENGTH,
    'maximum_target_token_length': MAXIMUM_TARGET_TOKEN_LENGTH,
    'number_of_spelling_candidates': NUMBER_OF_SPELLING_CANDIDATES,
    'training_data_source': 'Birkbeck Corpus + TOEFL-Spell (ETS)',
    'candidate_generation_method': 'SymSpell (Garbe 2018) + shuffle to prevent position bias',
}

evaluation_metrics_save_path = f'{final_model_save_directory}/evaluation_metrics.json'
with open(evaluation_metrics_save_path, 'w') as metrics_output_file:
    json.dump(reranker_evaluation_metrics, metrics_output_file, indent=2)

print(f'Final model saved to: {final_model_save_directory}')
print('\nSaved files:')
for saved_filename in os.listdir(final_model_save_directory):
    saved_file_size_kb = os.path.getsize(
        os.path.join(final_model_save_directory, saved_filename)
    ) / 1024
    print(f'  {saved_filename}  ({saved_file_size_kb:.1f} KB)')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/drive/MyDrive/SALITAyo_Reranker/final_model

Saved files:
  config.json  (1.5 KB)
  generation_config.json  (0.1 KB)
  model.safetensors  (236369.0 KB)
  tokenizer_config.json  (2.3 KB)
  tokenizer.json  (2058.0 KB)
  training_args.bin  (5.3 KB)
  evaluation_metrics.json  (0.5 KB)


**Expected output:**
```
Final model saved to: .../final_model

Saved files:
  config.json              (X.X KB)
  generation_config.json   (X.X KB)
  model.safetensors        (~230000 KB)  <- main model weights
  special_tokens_map.json  (X.X KB)
  spiece.model             (XXX.X KB)
  tokenizer.json           (XXX.X KB)
  tokenizer_config.json    (X.X KB)
  evaluation_metrics.json  (X.X KB)
```

In [ ]:
# ── CELL 16 ── Download all output files ─────────────────────────────────
import shutil
from google.colab import files

zip_archive_output_path = '/content/reranker_final'
shutil.make_archive(zip_archive_output_path, 'zip', reranker_output_directory)
files.download(f'{zip_archive_output_path}.zip')

total_output_size_mb = sum(
    os.path.getsize(os.path.join(directory_path, filename))
    for directory_path, subdirectory_names, filenames
    in os.walk(reranker_output_directory)
    for filename in filenames
) / (1024 * 1024)

print('Download initiated. Check your browser downloads folder.')
print(f'Total output size: {total_output_size_mb:.1f} MB')
print('\nIMPORTANT — After downloading, place files here on your laptop:')
print('  final_model/ folder -> backend/trained_models/reranker/final_model/')
print('  test_results_for_excel.csv -> for Excel evaluation workbook')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated. Check your browser downloads folder.
Total output size: 1622.6 MB

IMPORTANT — After downloading, place files here on your laptop:
  final_model/ folder -> backend/trained_models/reranker/final_model/
  test_results_for_excel.csv -> for Excel evaluation workbook


**Expected output:**
```
Download initiated.
Total output size: ~250-280 MB

IMPORTANT — After downloading, place files here on your laptop:
  final_model/ folder -> backend/trained_models/reranker/final_model/
  test_results_for_excel.csv -> for Excel evaluation workbook
```